# StackOverflow Preprocessing for QueryGPT (Phase 1)

Turn raw Parquet shards into **question → answer** sequences for decoder training.

| Step | Output |
|------|--------|
| Join Q&A (DuckDB) | `data/processed/qa_pairs.parquet` |
| Format + split | `qa_train.parquet`, `qa_val.parquet` |
| BPE + tokenize | `tokenizer/`, `tokenized/train.arrow`, `tokenized/val.arrow` |

Run from project root. Wikipedia / RAG chunking is **not** included (phase 2).

In [1]:
from pathlib import Path

# --- paths (relative to project root) ---
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_SO_DIR = PROJECT_ROOT / "data/raw/stackoverflow-posts"
PROCESSED_DIR = PROJECT_ROOT / "data/processed"
TOKENIZER_DIR = PROCESSED_DIR / "tokenizer"
TOKENIZED_DIR = PROCESSED_DIR / "tokenized"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZER_DIR.mkdir(parents=True, exist_ok=True)
TOKENIZED_DIR.mkdir(parents=True, exist_ok=True)

# --- preprocessing knobs ---
MAX_SHARDS = None          # None = all *.parquet in RAW_SO_DIR; set e.g. 2 for a quick test
MIN_ANSWER_CHARS = 32
MIN_QUESTION_CHARS = 10
VAL_FRACTION = 0.01
RANDOM_SEED = 42
MAX_SEQ_LEN = 1024
TOKENIZER_VOCAB_SIZE = 16000
TOKENIZER_SAMPLE_LINES = 200_000   # lines used to train BPE (increase for production)
ROW_BATCH_SIZE = 64            # CRITICAL: keep <=128 (Arrow scanner batch, not row groups)

SPECIAL_TOKENS = ["<pad>", "<unk>", "<bos>", "<eos>", "<sep>"]

# DuckDB memory tuning (join step only)
DUCKDB_MEMORY_LIMIT = "8GB"
DUCKDB_THREADS = 2
SKIP_QA_JOIN_IF_EXISTS = True
CLEANUP_JOIN_PARTS = True

# Heavy steps run via script (avoids Jupyter holding extra GB of RAM)
RUN_PREPROCESS_SCRIPT = True

print("Project root:", PROJECT_ROOT)
print("SO shards:", len(sorted(RAW_SO_DIR.glob("*.parquet"))))
print("Processed dir:", PROCESSED_DIR)

Project root: /Users/sbboss/NU/Q3/MSAI-495-2/Text-Project
SO shards: 20
Processed dir: /Users/sbboss/NU/Q3/MSAI-495-2/Text-Project/data/processed


## 1. Explore raw data

Post types: **1 = question**, **2 = answer**. Answers link to questions via `ParentId` (= question `Id`).

In [2]:
import pyarrow.parquet as pq
from collections import Counter

shard_paths = sorted(RAW_SO_DIR.glob("*.parquet"))
if MAX_SHARDS is not None:
    shard_paths = shard_paths[:MAX_SHARDS]

sample_path = shard_paths[0]
table = pq.read_table(sample_path, columns=["PostTypeId", "ParentId", "Title", "Body", "Tags", "Score"])
print("Sample shard:", sample_path.name, "| rows:", table.num_rows)
print("Columns:", table.column_names)
print("PostTypeId counts:", dict(Counter(table["PostTypeId"].to_pylist())))

# preview one question and one answer
for i in range(table.num_rows):
    if table["PostTypeId"][i].as_py() == 1:
        print("\n--- Question ---")
        print("Title:", (table["Title"][i].as_py() or "")[:200])
        print("Body:", (table["Body"][i].as_py() or "")[:200])
        break
for i in range(table.num_rows):
    if table["PostTypeId"][i].as_py() == 2:
        print("\n--- Answer ---")
        print("ParentId:", table["ParentId"][i].as_py())
        print("Body:", (table["Body"][i].as_py() or "")[:200])
        break

Sample shard: stackoverflow-posts-00000-of-00058.parquet | rows: 1000000
Columns: ['PostTypeId', 'ParentId', 'Title', 'Body', 'Tags', 'Score']
PostTypeId counts: {2: 753654, 1: 246346}

--- Question ---
Title: How do I calculate someone's age based on a DateTime type birthday?
Body: Given a `DateTime` representing a person's birthday, how do I calculate their age in years?


--- Answer ---
ParentId: 4
Body: An explicit cast to `double` like this isn't necessary:

```
double trans = (double) trackBar1.Value / 5000.0;
```


Identifying the constant as `5000.0` (or as `5000d`) is sufficient:

```
double tra


## 2. Join questions and answers (memory-safe)

**Do not** self-join the full dataset in one query (that reads ~20M posts twice and OOMs).

Instead:
1. **Pass 1** — extract questions shard-by-shard → `questions_index.parquet`
2. **Pass 2** — join each shard’s answers to that index → merge `qa_pairs.parquet`

In [4]:
import duckdb
from tqdm.auto import tqdm

qa_pairs_path = PROCESSED_DIR / "qa_pairs.parquet"
questions_path = PROCESSED_DIR / "questions_index.parquet"
parts_dir = PROCESSED_DIR / "_join_parts"
parts_dir.mkdir(parents=True, exist_ok=True)

if SKIP_QA_JOIN_IF_EXISTS and qa_pairs_path.exists():
    n_pairs = duckdb.sql(f"SELECT count(*) FROM read_parquet('{qa_pairs_path}')").fetchone()[0]
    print(f"Using existing qa_pairs.parquet ({n_pairs:,} rows)")
    if CLEANUP_JOIN_PARTS and parts_dir.exists():
        import shutil
        shutil.rmtree(parts_dir)
        print(f"Deleted join temp dir: {parts_dir}")
else:
    def configure_duckdb(con: duckdb.DuckDBPyConnection) -> None:
        con.execute(f"SET memory_limit='{DUCKDB_MEMORY_LIMIT}'")
        con.execute("SET preserve_insertion_order=false")
        con.execute(f"SET threads={DUCKDB_THREADS}")

    con = duckdb.connect()
    configure_duckdb(con)

    # --- Pass 1: question index (one shard at a time) ---
    if not questions_path.exists():
        print("Pass 1: building questions_index.parquet ...")
        q_parts = []
        for i, shard in enumerate(tqdm(shard_paths, desc="Questions")):
            q_part = parts_dir / f"questions_{i:03d}.parquet"
            con.execute(f"""
                COPY (
                    SELECT Id, Title, Body, Tags
                    FROM read_parquet('{shard}')
                    WHERE PostTypeId = 1
                      AND Body IS NOT NULL
                      AND length(trim(coalesce(Title, '')) || trim(Body)) >= {MIN_QUESTION_CHARS}
                ) TO '{q_part}' (FORMAT PARQUET)
            """)
            q_parts.append(str(q_part))
        q_glob = ", ".join(f"'{p}'" for p in q_parts)
        con.execute(f"""
            COPY (SELECT * FROM read_parquet([{q_glob}]))
            TO '{questions_path}' (FORMAT PARQUET)
        """)
        n_q = con.execute(f"SELECT count(*) FROM read_parquet('{questions_path}')").fetchone()[0]
        print(f"  Questions indexed: {n_q:,}")
    else:
        print(f"Pass 1 skipped — found {questions_path}")

    # --- Pass 2: join answers per shard (small peak memory) ---
    print("Pass 2: joining answers to questions (per shard) ...")
    qa_parts = []
    join_sql = """
        SELECT
            q.Id AS question_id,
            q.Title AS title,
            q.Body AS question_body,
            q.Tags AS tags,
            a.Id AS answer_id,
            a.Body AS answer_body,
            a.Score AS answer_score
        FROM read_parquet('{shard}') AS a
        INNER JOIN read_parquet('{questions}') AS q
            ON a.ParentId = q.Id
        WHERE a.PostTypeId = 2
          AND a.Body IS NOT NULL
          AND length(trim(a.Body)) >= {min_answer}
    """
    for i, shard in enumerate(tqdm(shard_paths, desc="Join shards")):
        qa_part = parts_dir / f"qa_{i:03d}.parquet"
        con.execute(
            f"COPY ({join_sql.format(shard=shard, questions=questions_path, min_answer=MIN_ANSWER_CHARS)}) "
            f"TO '{qa_part}' (FORMAT PARQUET)"
        )
        qa_parts.append(str(qa_part))

    qa_glob = ", ".join(f"'{p}'" for p in qa_parts)
    con.execute(f"""
        COPY (SELECT * FROM read_parquet([{qa_glob}]))
        TO '{qa_pairs_path}' (FORMAT PARQUET)
    """)
    con.close()

    n_pairs = duckdb.sql(f"SELECT count(*) FROM read_parquet('{qa_pairs_path}')").fetchone()[0]
    print(f"Saved {n_pairs:,} Q&A pairs -> {qa_pairs_path}")

    if CLEANUP_JOIN_PARTS and parts_dir.exists():
        import shutil
        shutil.rmtree(parts_dir)
        print(f"Deleted join temp dir: {parts_dir} (frees ~50GB)")

Saved 12,791,691 Q&A pairs -> /Users/sbboss/NU/Q3/MSAI-495-2/Text-Project/data/processed/qa_pairs.parquet


## 3. Tokenize

| Mode | When |
|------|------|
| **Fast** (`preprocess_stackoverflow_fast.py`) | Cluster / lots of RAM — DuckDB split + 32-way `encode_batch` |
| **Stream** (`preprocess_stackoverflow_stream.py`) | Laptop / low RAM — tiny batches |

Outputs: `tokenized/train/` and `tokenized/val/` (HuggingFace `save_to_disk` format).

In [ ]:
import subprocess
import sys

USE_FAST_PREPROCESS = True  # False = low-RAM stream script

if USE_FAST_PREPROCESS:
    script = PROJECT_ROOT / "scripts/preprocess_stackoverflow_fast.py"
    import os
    workers = max(4, (os.cpu_count() or 8) - 2)
    cmd = [
        sys.executable, str(script),
        "--processed-dir", str(PROCESSED_DIR),
        "--memory-limit", "80GB",
        "--workers", str(workers),
        "--encode-batch-size", "4096",
        "--max-seq-len", str(MAX_SEQ_LEN),
        "--vocab-size", str(TOKENIZER_VOCAB_SIZE),
        "--cleanup-parts",
    ]
else:
    script = PROJECT_ROOT / "scripts/preprocess_stackoverflow_stream.py"
    cmd = [
        sys.executable, str(script),
        "--processed-dir", str(PROCESSED_DIR),
        "--row-batch-size", str(ROW_BATCH_SIZE),
        "--max-seq-len", str(MAX_SEQ_LEN),
        "--vocab-size", str(TOKENIZER_VOCAB_SIZE),
        "--tokenizer-sample-rows", "50000",
        "--cleanup-parts",
    ]

print("Run in terminal (recommended):")
print(" ", " ".join(cmd))
print()

if RUN_PREPROCESS_SCRIPT:
    proc = subprocess.run(cmd, cwd=str(PROJECT_ROOT))
    if proc.returncode != 0:
        raise RuntimeError(f"Script failed with code {proc.returncode}")
else:
    print("Set RUN_PREPROCESS_SCRIPT=True or run the command above in a terminal.")

## 4. Done — check outputs

Old notebook cells below are **deprecated** (they OOM). Use the script only.

In [ ]:
manifest = PROCESSED_DIR / "tokenized" / "manifest.json"
if manifest.exists():
    print(manifest.read_text())
    print("Train chunks:", len(list((PROCESSED_DIR / "tokenized/train").glob("chunk_*.parquet"))))
    print("Val chunks:", len(list((PROCESSED_DIR / "tokenized/val").glob("chunk_*.parquet"))))
else:
    print("Run section 3 first.")

## 5. Free disk (optional)

In [ ]:
import shutil

for path in [PROCESSED_DIR / "_join_parts", PROCESSED_DIR / "questions_index.parquet"]:
    if path.exists():
        shutil.rmtree(path) if path.is_dir() else path.unlink()
        print("Removed:", path)

In [ ]:
# Deprecated — use scripts/preprocess_stackoverflow_stream.py

## 7. Free disk (run after join + tokenize)

| Path | Safe to delete? |
|------|------------------|
| `_join_parts/` | Yes (~50GB) after `qa_pairs.parquet` exists |
| `questions_index.parquet` | Yes after join (only needed for join) |
| `qa_train.parquet` / `qa_val.parquet` | Skip creating them with `SKIP_TEXT_PARQUET=True` |

## 8. Summary

| Artifact | Purpose |
|----------|---------|
| `qa_pairs.parquet` | Joined raw Q&A |
| `tokenizer/tokenizer.json` | BPE vocab |
| `tokenized/train/chunk_*.parquet` | `input_ids` shards for JAX |
| `tokenized/manifest.json` | paths + stats |

**Next:** JAX/Flax training loop reading `tokenized/*.parquet` in batches.

In [ ]:
import shutil

# Manual cleanup — run once to reclaim disk
for path in [PROCESSED_DIR / "_join_parts", PROCESSED_DIR / "questions_index.parquet"]:
    if path.exists():
        if path.is_dir():
            shutil.rmtree(path)
        else:
            path.unlink()
        print("Removed:", path)
    else:
        print("Already gone:", path)